alternative LSH approach: frame2frame

In [5]:
from lshashpy3 import LSHash
import pandas as pd
import numpy as np
import sys
np.set_printoptions(threshold=sys.maxsize)

In [6]:
# setting up LSH instances
# frame --> frame

# Load in frame_training_data.csv (product of preprocess_for_LSH.ipynb)

data = pd.read_csv("frame_training_data.csv")

flat_poses_first10 = {}
flat_poses_pseudo = {}

for i, row in data.iterrows():
    if row['LemmaID'] not in flat_poses_first10.keys():
        flat_poses_first10[row["LemmaID"]] = []
    if row['LemmaID'] not in flat_poses_pseudo.keys():
        flat_poses_pseudo[row["LemmaID"]] = []
    points_pseudo = row['flat_pseudo_all'].strip("[]").split(" ")
    points_first10 = row['flat_10frames'].strip("[]").split(" ")
    add_to_flat_poses_pseudo = []
    add_to_flat_poses_first10 = []
    for j in points_pseudo:
        if j != "[" and j!= "]" and j != "":
            add_to_flat_poses_pseudo.append(float(j))
    for k in points_first10:
        if k != "[" and k!= "]" and k != "":
            add_to_flat_poses_first10.append(float(k))
    flat_poses_first10[row['LemmaID']].append(np.array(add_to_flat_poses_first10))
    flat_poses_pseudo[row['LemmaID']].append(np.array(add_to_flat_poses_pseudo))
    # labels.append(row['LemmaID'])

# print(flat_poses_first10)
# print(len(flat_poses_first10['hamburger'][0])) # have to index 0 for each key to get to it!

# len of each item in flat_poses_first10 = 990; aka each item in flat_poses_first10 is all the data from one video
# split each video in flat_poses_first10 into 10 frames; divide by 99 (num data points/frame)
# each frame is going to need a tag for lemma


def get_frameByFrame(d): # input: dict
    label_list = []
    frame_list = []
    for key, value in d.items():
        # add labels corresponding to # of frames in vid; add frame index to end!
        num_frames = int(len(d[key][0])/99)
        # print("num frames: ",num_frames)
        for i in range(num_frames):
            label = key + str(i)
            label_list.append(label)
            # i+=1
        # add 99 data points to current frame, then start over
        for i in range(0, len(d[key][0]), 99):
            # Slice the original list from index 'i' up to index 'i + 99'
            # and append this slice (which is a new list of up to 5 items)
            # into the new list.
            current_frame = d[key][0][i:i + 99]
            frame_list.append(current_frame)
        # print("len frame list: ", len(frame_list))
    return label_list, frame_list

first10_labels, first10_toIndex = get_frameByFrame(flat_poses_first10)
pseudo_labels, pseudo_toIndex = get_frameByFrame(flat_poses_pseudo)
# print(len(first10_labels))
# print(len(first10_toIndex))
# print(pseudo_labels)


# set up LSH!
d = 99

lsh_first10 = LSHash(hash_size=5, input_dim=d, num_hashtables=5, overwrite=False)
lsh_pseudo = LSHash(hash_size=5, input_dim=d, num_hashtables=5,overwrite=False)

# batch index
for i in range(len(first10_labels)):
    lsh_first10.index(first10_toIndex[i], first10_labels[i])
for i in range(len(pseudo_labels)):
    lsh_pseudo.index(pseudo_toIndex[i], pseudo_labels[i])

# top_n = 3
# nn = lsh.query([10,12,99,1,5,30,1,1], num_results=top_n, distance_func="euclidean")
# print(nn)

In [7]:
# Function for analyzing first 10 frames

def test_first10(coord_file): # input is coordinate file from user input video
    correct_label = coord_file[:-17] # exclude "_pose-results.txt"
    print("correct label: ", correct_label)
    test_coords = pd.read_csv(coord_file, names=['pose_index', 'x', 'y', 'z'])
    test_10frames = []

    j = 0 # use to get first 10 of user input (test) vid
    for i, row in test_coords.iterrows():
        test_10frames.append(float(row['x']))
        test_10frames.append(float(row['y']))
        test_10frames.append(float(row['z']))
        j += 1
        if j == 330:
            break
    
    # split test_10frames list into list of lists (each frame gets its own inner list!)
    frame2frame = []
    for i in range(0, len(test_10frames), 99):
        # Slice the original list from index 'i' up to index 'i + 99'
        # and append this slice (which is a new list of up to 5 items)
        # into the new list.
        current_frame = test_10frames[i:i + 99]
        frame2frame.append(current_frame)

    # print("frame2frame: ", frame2frame)

    # turn to array & query LSH
    top_n = 5
    frame2frame_matches = []
    for frame in frame2frame:
        frame_array = np.asarray(frame)
        query_vector = frame_array
        nn5_first10 = lsh_first10.query(query_vector, num_results=top_n, distance_func="euclidean")
        # get labels
        frame_matches = []
        for i in range(len(nn5_first10)):
            frame_matches.append(nn5_first10[i][0][1])
        frame2frame_matches.append(frame_matches)
    

    # evaluate
    # rank choice voting
    # who gets the most top votes across frames?
    match_counts = {"first": {}, "other":{}, "overall":{}}
    for match_group in frame2frame_matches:
        for i in range(len(match_group)):
            match = match_group[i][:-1]
            if i == 0:
                if match not in match_counts["first"].keys():
                    match_counts['first'][match] = 0
                match_counts['first'][match] += 1
            else:
                if match not in match_counts['other'].keys():
                    match_counts['other'][match] = 0
                match_counts['other'][match] += 1
            if match not in match_counts['overall'].keys():
                match_counts['overall'][match] = 0
            match_counts['overall'][match] += 1
    
    print("match counts: ", match_counts)

    # evaluate match_counts!

    # first first 
    max_first = max(match_counts['first'].values())
    # print(max_first)
    firsts = [key for key, value in match_counts['first'].items() if value == max_first]

    # first overall
    max_overall_first = max(match_counts['overall'].values())
    overall_firsts = [key for key, value in match_counts['overall'].items() if value == max_overall_first]

    # second overall
    # 1. Extract values and convert to a set to remove duplicates
    unique_overall_values = set(match_counts['overall'].values())
    # 2. Sort the unique values in ascending order
    sorted_overall_values = sorted(unique_overall_values)
    # 3. Get the second-to-last element (the second maximum)
    if len(sorted_overall_values) >= 2:
        max_overall_second = sorted_overall_values[-2]
        overall_seconds = [key for key, value in match_counts['overall'].items() if value == max_overall_second]
    else:
        max_overall_second = "N/A"
        overall_seconds = "N/A"

    # print(firsts, overall_firsts, overall_seconds)

    # evaluate that?
    print(frame2frame_matches)

    return correct_label, frame2frame_matches, firsts, max_first, overall_firsts, max_overall_first, overall_seconds, max_overall_second

In [8]:
test_correct_label, test_matches, test_firsts, test_first_votes, test_overall_firsts, test_overall_first_votes, test_overall_seconds, test_overall_second_votes = test_first10('chocolate-me_pose-results.txt')
print(test_matches)
print("received most votes for most likely: ",test_firsts, test_first_votes)
print("received most overall votes: ", test_overall_firsts, test_overall_first_votes)
print("received second-most overall votes: ",test_overall_seconds, test_overall_second_votes)

correct label:  chocolate-me
match counts:  {'first': {'coffee': 10}, 'other': {'butter': 33, 'coffee': 7}, 'overall': {'coffee': 17, 'butter': 33}}
[['coffee0', 'butter5', 'butter8', 'butter6', 'butter7'], ['coffee0', 'butter8', 'butter5', 'butter9', 'coffee1'], ['coffee0', 'butter8', 'butter5', 'coffee1', 'butter9'], ['coffee0', 'butter8', 'coffee1', 'butter5', 'butter9'], ['coffee0', 'butter8', 'coffee1', 'butter5', 'butter9'], ['coffee0', 'butter8', 'coffee1', 'butter5', 'butter9'], ['coffee0', 'butter5', 'coffee1', 'butter7', 'butter6'], ['coffee0', 'butter5', 'coffee1', 'butter6', 'butter7'], ['coffee0', 'butter5', 'butter8', 'butter6', 'butter7'], ['coffee0', 'butter5', 'butter8', 'butter6', 'butter9']]
[['coffee0', 'butter5', 'butter8', 'butter6', 'butter7'], ['coffee0', 'butter8', 'butter5', 'butter9', 'coffee1'], ['coffee0', 'butter8', 'butter5', 'coffee1', 'butter9'], ['coffee0', 'butter8', 'coffee1', 'butter5', 'butter9'], ['coffee0', 'butter8', 'coffee1', 'butter5', 'butte

In [9]:
# run on relevant files, add results to df-->csv
frame2frame_first10_results_df = pd.DataFrame(columns=['correct_label', 'match_counts', "first-first", "first-first_votes", "first-overall", "first-overall_votes", "second-overall", "second-overall_votes"])

In [10]:
# Function for analyzing pseudo-keyframes

def test_pseudo(coord_file): # input is coordinate file from user input video
    correct_label = coord_file[:-17] # exclude "_pose-results.txt"
    print("correct label: ", correct_label)
    test_coords_df = pd.read_csv(coord_file, names=['pose_index', 'x', 'y', 'z'])
    test_coords = [] # make it into a list
    for i, row in test_coords_df.iterrows():
        test_coords.append(float(row['x']))
        test_coords.append(float(row['y']))
        test_coords.append(float(row['z']))
    test_pseudokeys = []

    # get pseudo-keyframes out of user input (test) vid
    for i in range(0, len(test_coords), 330):
        current_frame = test_coords[i:i+330]
        test_pseudokeys.append(current_frame)
    # for item in test_pseudokeys:
    #     if len(item) != 330:
    #         test_pseudokeys.remove(item)
    test_pseudokeys = [
    item for item in test_pseudokeys
    if len(item) == 330 ]
    all_coords = [] # make into 1-d list of coordinates
    for item in test_pseudokeys:
        for item2 in item:
            all_coords.append(item2)

    
    # print("test pseudokeys: ", test_pseudokeys)
    # print("all coords: ", all_coords)
    
    # split all_coords list into list of lists (each frame - len 99 - gets its own inner list!)
    frame2frame = []
    for i in range(0, len(all_coords), 99):
        # Slice the original list from index 'i' up to index 'i + 99'
        # and append this slice (which is a new list of up to 5 items)
        # into the new list.
        current_frame = all_coords[i:i + 99]
        frame2frame.append(current_frame)

    # turn to array & query LSH
    top_n = 5
    frame2frame_matches = []
    for frame in frame2frame:
        frame_array = np.asarray(frame)
        query_vector = frame_array
        nn5_pseudo = lsh_pseudo.query(query_vector, num_results=top_n, distance_func="euclidean")
        # get labels
        frame_matches = []
        for i in range(len(nn5_pseudo)):
            frame_matches.append(nn5_pseudo[i][0][1])
        frame2frame_matches.append(frame_matches)
    

    # evaluate
    # rank choice voting
    # who gets the most top votes across frames?
    match_counts = {"first": {}, "other":{}, "overall":{}}
    for match_group in frame2frame_matches:
        for i in range(len(match_group)):
            match = match_group[i][:-1]
            if i == 0:
                if match not in match_counts["first"].keys():
                    match_counts['first'][match] = 0
                match_counts['first'][match] += 1
            else:
                if match not in match_counts['other'].keys():
                    match_counts['other'][match] = 0
                match_counts['other'][match] += 1
            if match not in match_counts['overall'].keys():
                match_counts['overall'][match] = 0
            match_counts['overall'][match] += 1
    
    print("match counts: ", match_counts)

    # evaluate match_counts!

    # first first 
    max_first = max(match_counts['first'].values())
    # print(max_first)
    firsts = [key for key, value in match_counts['first'].items() if value == max_first]

    # first overall
    max_overall_first = max(match_counts['overall'].values())
    overall_firsts = [key for key, value in match_counts['overall'].items() if value == max_overall_first]

    # second overall
    # 1. Extract values and convert to a set to remove duplicates
    unique_overall_values = set(match_counts['overall'].values())
    # 2. Sort the unique values in ascending order
    sorted_overall_values = sorted(unique_overall_values)
    # 3. Get the second-to-last element (the second maximum)
    if len(sorted_overall_values) >= 2:
        max_overall_second = sorted_overall_values[-2]
        overall_seconds = [key for key, value in match_counts['overall'].items() if value == max_overall_second]
    else:
        max_overall_second = "N/A"
        overall_seconds = "N/A"

    # print(firsts, overall_firsts, overall_seconds)

    # evaluate that?

    return correct_label, frame2frame_matches, firsts, max_first, overall_firsts, max_overall_first, overall_seconds, max_overall_second

In [11]:
test_correct_label, test_matches, test_firsts, test_first_votes, test_overall_firsts, test_overall_first_votes, test_overall_seconds, test_overall_second_votes = test_pseudo('chocolate-me_pose-results.txt')
print(test_matches)
print("received most votes for most likely: ",test_firsts, test_first_votes)
print("received most overall votes: ", test_overall_firsts, test_overall_first_votes)
print("received second-most overall votes: ",test_overall_seconds, test_overall_second_votes)

correct label:  chocolate-me
match counts:  {'first': {'and': 139, 'coffee': 1}, 'other': {'coffee': 143, 'butter': 60, 'and': 354, 'chair': 3}, 'overall': {'and': 493, 'coffee': 144, 'butter': 60, 'chair': 3}}
[['and9', 'coffee0', 'butter0', 'coffee1', 'and2'], ['coffee0', 'and9', 'coffee1', 'butter0', 'coffee8'], ['and9', 'coffee0', 'coffee1', 'butter0', 'coffee8'], ['and9', 'coffee0', 'coffee1', 'butter0', 'coffee8'], ['and9', 'coffee0', 'coffee1', 'butter0', 'coffee8'], ['and9', 'coffee0', 'coffee1', 'butter0', 'coffee8'], ['and9', 'coffee0', 'and2', 'butter1', 'coffee1'], ['and9', 'coffee0', 'and2', 'butter1', 'coffee1'], ['and9', 'coffee0', 'butter0', 'coffee1', 'and2'], ['and9', 'coffee0', 'butter0', 'coffee1', 'and2'], ['and9', 'coffee0', 'coffee1', 'butter0', 'and2'], ['and9', 'coffee0', 'and2', 'coffee1', 'butter0'], ['and9', 'coffee0', 'and2', 'coffee1', 'butter0'], ['and9', 'coffee0', 'and2', 'butter1', 'coffee1'], ['and9', 'and2', 'coffee0', 'butter1', 'and1'], ['and9', 'a

In [12]:
# run on relevant files, save to df-->csv
frame2frame_pseudo_results_df = pd.DataFrame(columns=['correct_label', 'match_counts', "first-first", "first-first_votes", "first-overall", "first-overall_votes", "second-overall", "second-overall_votes"])